In [1]:
import os
import sys
import pandas as pd

current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    if project_root not in sys.path:
        sys.path.insert(0, project_root)
        print(f"✅ 已添加项目根目录: {project_root}")

from config import get_driver
driver = get_driver()
import warnings
warnings.filterwarnings("ignore")

# 全局 context：跨 Cell 共享对象 ID，避免硬编码
context = {
    "pathogen_ids": {},
    "source_ids": {},
    "org_ids": {},
    "program_ids": {},
    "event_ids": {},
    "strategy_ids": {},
    "construct_ids": {},
    "claim_ids": {},
    "result_ids": {},
    "tech_assessment_ids": {},
    "comp_assessment_ids": {},
    "brief_ids": {},
    "decision_ids": [],
    "use_event_ids": [],
}
from src.shared.init_schema import initialize_neo4j_schema
initialize_neo4j_schema(driver)

print("✅ 模块导入完成")

✅ 已添加项目根目录: D:\IdeaProjects\vs\phage-workspace-mvp
✅ Neo4j Schema 初始化完成（18 条约束, 9 个索引, 幂等）
✅ 模块导入完成


In [2]:
# 测试 CI 服务 - 创建 Proteon 组织、项目、事件（含来源和病原体）
from config import get_driver
from src.ci.organization_service import create_organization
from src.ci.program_service import create_development_program
from src.ci.event_service import capture_intelligence_event
from src.shared.source_artifact_service import create_source_artifact

driver = get_driver()

org_id = create_organization(
    driver,
    canonical_name="Proteon Pharmaceuticals",
    organization_type="biotech",
    aliases=["Proteon"],
    headquarters_country="Poland",
    website="https://www.proteonpharma.com",
    description="Developing bacteriophage-based solutions for animal health and food safety, targeting poultry and swine bacterial infections."
)
context["org_ids"]["Proteon Pharmaceuticals"] = org_id
print(f"✅ 组织创建成功: {org_id}")

# 创建项目并指定靶向病原体
prog_id = create_development_program(
    driver,
    organization_id=org_id,
    canonical_name="BAFASAL",
    program_type="therapeutic",
    development_stage="commercial",
    modality="cocktail",
    target_pathogen_species=["Salmonella enterica"],  # ✅ 指定真实物种
)
context["program_ids"]["BAFASAL"] = prog_id
print(f"✅ 项目创建成功: {prog_id}")

# 创建两个 SourceArtifact（用于事件来源）
src1 = create_source_artifact(
    driver,
    source_type="regulatory_filing",
    title="BAFASAL EU Authorization — EMA Decision",
    url="https://www.ema.europa.eu/...",
    published_date="2022-03-15",
    credibility_tier="primary",
    actor_id="test_user"
)
context["source_ids"]["BAFASAL_EMA"] = src1
print(f"✅ SourceArtifact 1 创建成功: {src1}")

src2 = create_source_artifact(
    driver,
    source_type="press_release",
    title="Proteon Series B funding announcement",
    url="https://www.proteonpharma.com/news/series-b",
    published_date="2023-05-10",
    credibility_tier="secondary",
    actor_id="test_user"
)
context["source_ids"]["Proteon_SeriesB"] = src2
print(f"✅ SourceArtifact 2 创建成功: {src2}")

# 事件1（欧盟授权）关联 src1
event_id_1 = capture_intelligence_event(
    driver,
    event_type="regulatory_update",
    title="BAFASAL® receives EU authorization for Salmonella control in poultry",
    factual_summary="Proteon Pharmaceuticals' BAFASAL® product received EU authorization as a feed additive for reducing Salmonella in poultry, the first phage-based solution approved in Europe for this indication.",
    organization_id=org_id,
    program_id=prog_id,
    event_date="2022-03-15",
    published_at="2022-03-16",
    source_ids=[src1],  # ✅ 传入来源
    actor_id="test_user"
)
context["event_ids"]["BAFASAL EU authorization"] = event_id_1
print(f"✅ 事件1创建成功: {event_id_1}")

# 事件2（融资）关联 src2
event_id_2 = capture_intelligence_event(
    driver,
    event_type="funding",
    title="Proteon Pharmaceuticals secures €15M Series B funding",
    factual_summary="Proteon Pharmaceuticals closed a €15 million Series B financing round led by existing investors to expand its phage-based product portfolio and enter new markets.",
    organization_id=org_id,
    program_id=prog_id,
    event_date="2023-05-10",
    published_at="2023-05-11",
    source_ids=[src2],  # ✅ 传入来源
    actor_id="test_user"
)
context["event_ids"]["Proteon Series B"] = event_id_2
print(f"✅ 事件2创建成功: {event_id_2}")

print("\n🔗 验证组织-项目-事件关系：")
with driver.session() as session:
    result = session.run("""
        MATCH (o:Organization)-[:DEVELOPS]->(p:DevelopmentProgram)<-[:AFFECTS]-(e:IntelligenceEvent)
        WHERE o.organization_id = $oid
        RETURN o.canonical_name AS org, p.canonical_name AS prog, e.title AS event, e.event_type AS type
        ORDER BY e.event_date
    """, oid=org_id)
    for record in result:
        print(f"  组织: {record['org']}")
        print(f"  项目: {record['prog']}")
        print(f"  事件: {record['event']} ({record['type']})")
        print("  ---")

✅ 组织创建成功: CI:ORG:3AB2C864
✅ 项目创建成功: CI:PROG:E5FB7E35
✅ SourceArtifact 1 创建成功: SRC:585AE14F
✅ SourceArtifact 2 创建成功: SRC:E2DF824F
✅ 事件1创建成功: CI:EVT:617F665B
✅ 事件2创建成功: CI:EVT:B1A397EB

🔗 验证组织-项目-事件关系：
  组织: Proteon Pharmaceuticals
  项目: BAFASAL
  事件: BAFASAL® receives EU authorization for Salmonella control in poultry (regulatory_update)
  ---
  组织: Proteon Pharmaceuticals
  项目: BAFASAL
  事件: Proteon Pharmaceuticals secures €15M Series B funding (funding)
  ---


In [3]:
from config import get_driver
from src.ci.competitor_profile import build_competitor_profile, list_organizations

driver = get_driver()

orgs = list_organizations(driver)
print("📋 可用组织列表：")
for org in orgs:
    print(f"   {org['id']} - {org['name']} ({org['type']})")

if orgs:
    org_id = orgs[0]['id']
    print(f"\n📊 生成 {orgs[0]['name']} 的完整档案：\n" + "="*60)
    
    profile = build_competitor_profile(driver, org_id)
    
    print(f"📌 基本信息")
    print(f"   名称: {profile['organization']['name']}")
    print(f"   类型: {profile['organization']['org_type']}")
    print(f"   国家: {profile['organization']['country']}")
    print(f"   描述: {profile['organization']['description']}")
    
    print(f"\n📦 研发项目 ({len(profile['active_programs'])} 个)")
    for prog in profile['active_programs']:
        pathogens = ', '.join([p['species'] for p in prog.get('target_pathogens', []) if p.get('species')])
        print(f"   - {prog['name']} ({prog['stage']}) -> 靶向: {pathogens or '未指定'}")
    
    print(f"\n📰 情报事件 ({len(profile['recent_events'])} 条)")
    for evt in profile['recent_events'][:3]:
        print(f"   - {evt['event_date']} | {evt['event_type']} | {evt['title'][:40]}...")
    
    print(f"\n⚠️ 数据缺口:")
    for gap in profile['data_gaps']:
        print(f"   - {gap}")
    
    print(f"\n📅 数据截止: {profile['as_of_date']}")

📋 可用组织列表：
   CI:ORG:31A321BD - Adaptive Phage Therapeutics (biotech)
   CI:ORG:1D0D468D - Armata Pharmaceuticals (biotech)
   CI:ORG:D2B4C6DE - BiomX (biotech)
   CI:ORG:86AA64E6 - Intralytix (biotech)
   CI:ORG:48615D48 - Locus Biosciences (biotech)
   CI:ORG:91D05176 - PHAXIAM Therapeutics (biotech)
   CI:ORG:FCABF250 - Phagelux (biotech)
   CI:ORG:3AB2C864 - Proteon Pharmaceuticals (biotech)
   CI:ORG:44078922 - 创噬纪 (biotech)
   CI:ORG:C66F1D9C - 格瑞农生物 (biotech)
   CI:ORG:50563C50 - 青岛诺安百特 (biotech)

📊 生成 Adaptive Phage Therapeutics 的完整档案：
📌 基本信息
   名称: Adaptive Phage Therapeutics
   类型: biotech
   国家: USA
   描述: 临床阶段生物技术公司，拥有PhageBank™噬菌体库平台，针对多重耐药菌感染[reference:0][reference:1]

📦 研发项目 (3 个)
   - PhageBank-CF (phase_1_2) -> 靶向: 未指定
   - PhageBank-PJI (phase_1_2) -> 靶向: 未指定
   - PhageBank-DFO (phase_1_2) -> 靶向: 未指定

📰 情报事件 (2 条)
   - 2021-01-15 | regulatory_update | APT announces FDA IND clearance for Phag...
   - 2019-06-01 | funding | APT awarded $14M DoD contract...

⚠️ 数据缺口:
   -

In [4]:
from config import get_driver
from src.ci.organization_service import detect_material_changes, get_organizations_with_recent_changes
from datetime import datetime, timedelta

driver = get_driver()

# 1. 查看哪些组织近期有变化（放宽到365天）
print("📊 近期有变化的组织（最近365天）：")
active_orgs = get_organizations_with_recent_changes(driver, days_back=365, min_changes=1)
if active_orgs:
    for org in active_orgs:
        print(f"   {org['name']} - {org['event_count']} 个事件")
else:
    print("   ⚠️ 没有组织有近期事件，将使用第一个组织进行演示")

# 2. 选择一个组织进行分析
# 如果 active_orgs 为空，就手动指定一个组织ID（从 list_organizations 获取）
if active_orgs:
    org_id = active_orgs[0]['id']
    org_name = active_orgs[0]['name']
else:
    # 从数据库中获取第一个组织作为备选
    from src.ci.competitor_profile import list_organizations
    all_orgs = list_organizations(driver)
    if all_orgs:
        org_id = all_orgs[0]['id']
        org_name = all_orgs[0]['name']
        print(f"   ℹ️ 使用组织: {org_name}（无近期事件，仅用于演示结构）")
    else:
        print("❌ 数据库中没有组织，请先导入数据")
        # 如果确实没有数据，可以跳过后续代码
        org_id = None

if org_id:
    print(f"\n🔍 检测 {org_name} 的重大变化（最近365天）：\n" + "="*60)
    
    changes = detect_material_changes(
        driver, 
        organization_id=org_id,
        days_back=365  # 改为365天
    )
    
    print(f"📌 统计：")
    print(f"   - 新事件: {changes['total_new_events']} 条")
    print(f"   - 新项目: {changes['total_new_programs']} 个")
    print(f"   - 状态变化: {changes['total_status_changes']} 次")
    print(f"   - 高影响事件: {changes['total_high_impact_events']} 条")
    print(f"   - 有重大变化: {'✅ 是' if changes['has_material_change'] else '❌ 否'}")
    
    if changes['changes']['new_events']:
        print(f"\n📰 最新事件：")
        for evt in changes['changes']['new_events'][:3]:
            print(f"   - {evt['event_date']} | {evt['event_type']} | {evt['title'][:40]}...")
    
    if changes['changes']['high_impact_events']:
        print(f"\n⚡ 高影响事件：")
        for evt in changes['changes']['high_impact_events'][:3]:
            print(f"   - {evt['event_date']} | {evt['event_type']} | {evt['title'][:40]}...")
    
    if changes['changes']['new_programs']:
        print(f"\n🆕 新项目：")
        for prog in changes['changes']['new_programs']:
            print(f"   - {prog['name']} ({prog['stage']})")

# 3. 手动指定日期检测（可以直接用组织ID）
if org_id:
    print("\n" + "="*60)
    print("🔍 自定义日期检测（自 2026-01-01 以来）：")
    custom_changes = detect_material_changes(
        driver,
        organization_id=org_id,
        since_date="2026-01-01"
    )
    print(f"   发现 {custom_changes['total_new_events']} 个新事件")
    
    if custom_changes['has_material_change']:
        print("   ✅ 有重大变化")
        for evt in custom_changes['changes']['new_events'][:2]:
            print(f"      - {evt['title'][:50]}...")
    else:
        print("   ℹ️ 此时间段内无变化")

📊 近期有变化的组织（最近365天）：
   BiomX - 2 个事件
   格瑞农生物 - 1 个事件

🔍 检测 BiomX 的重大变化（最近365天）：
📌 统计：
   - 新事件: 2 条
   - 新项目: 3 个
   - 状态变化: 0 次
   - 高影响事件: 1 条
   - 有重大变化: ✅ 是

📰 最新事件：
   - 2026-02-03 | partnership | BiomX acquires Adaptive Phage Therapeuti...
   - 2025-11-04 | regulatory_update | BiomX receives positive FDA feedback for...

⚡ 高影响事件：
   - 2026-02-03 | partnership | BiomX acquires Adaptive Phage Therapeuti...

🆕 新项目：
   - BX004 (phase_2b)
   - BX011 (preclinical)
   - BX211 (phase_2)

🔍 自定义日期检测（自 2026-01-01 以来）：
   发现 1 个新事件
   ✅ 有重大变化
      - BiomX acquires Adaptive Phage Therapeutics...


In [5]:
# 测试工程化噬菌体情报：工程策略 + 构建体
from config import get_driver
from src.engineering_intelligence.strategy_classifier import create_engineering_strategy, get_all_strategies
from src.engineering_intelligence.construct_service import create_engineered_construct, get_constructs_by_strategy

driver = get_driver()

print("🔬 创建工程策略...")
strategy_id = create_engineering_strategy(
    driver,
    strategy_type="host_range_expansion",
    description="通过改造尾纤维或受体结合蛋白来扩展宿主范围",
    evidence_maturity="in_vitro"
)
context["strategy_ids"]["host_range_expansion"] = strategy_id
print(f"   ✅ 策略创建成功: {strategy_id}")

strategy_id_2 = create_engineering_strategy(
    driver,
    strategy_type="lysis_enhancement",
    description="增强裂解活性，优化裂解模块",
    evidence_maturity="in_vitro"
)
context["strategy_ids"]["lysis_enhancement"] = strategy_id_2
print(f"   ✅ 策略创建成功: {strategy_id_2}")

print("\n📋 所有工程策略：")
strategies = get_all_strategies(driver)
for s in strategies:
    print(f"   - {s['strategy_type']} (成熟度: {s['evidence_maturity']})")

print("\n🧬 创建工程化构建体...")
construct_id = create_engineered_construct(
    driver,
    public_name="vB_Kpn_HRE_001",
    construct_code="HRE-001",
    parent_phage_name="PKP001",
    intended_effects=["宿主范围扩展", "针对KL47型肺炎克雷伯菌"],
    target_pathogen_ids=["PATH-003"],  # 导入时已创建
    strategy_ids=[strategy_id],
    construct_status="in_vitro_tested",
    first_public_date="2025-06-15"
)
context["construct_ids"]["vB_Kpn_HRE_001"] = construct_id
print(f"   ✅ 构建体创建成功: {construct_id}")

print(f"\n🔍 查询策略 'host_range_expansion' 下的构建体：")
constructs = get_constructs_by_strategy(driver, strategy_id)
for c in constructs:
    print(f"   - {c['name']} ({c['status']}) -> 亲本: {c['parent_phage']} -> 靶向: {c['target_pathogens']}")

print("\n🎉 工程化噬菌体情报模块初始化完成！")

🔬 创建工程策略...
   ✅ 策略创建成功: ENG:STRAT:907D832A
   ✅ 策略创建成功: ENG:STRAT:1516B23F

📋 所有工程策略：
   - host_range_expansion (成熟度: in_vitro)
   - lysis_enhancement (成熟度: in_vitro)

🧬 创建工程化构建体...
   ✅ 构建体创建成功: ENG:CONST:12FCFA84

🔍 查询策略 'host_range_expansion' 下的构建体：
   - vB_Kpn_HRE_001 (in_vitro_tested) -> 亲本: PKP001 -> 靶向: ['Klebsiella pneumoniae']

🎉 工程化噬菌体情报模块初始化完成！


In [6]:
# 测试 TechnicalClaim 和 TechnicalResult
from config import get_driver
from src.engineering_intelligence.claim_extractor import (
    create_technical_claim,
    create_technical_result,
    get_claims_by_construct,
    get_results_by_construct,
    detect_claim_evidence_gaps
)

driver = get_driver()

# 从 context 获取构建体 ID，例如："ENG:CONST:39845AFF"
construct_id = context["construct_ids"].get("vB_Kpn_HRE_001")
if construct_id is None:
    raise ValueError("构建体 vB_Kpn_HRE_001 未创建，请先执行 Cell 5")

print("📝 创建技术主张（Claim）...")

claim_id_1 = create_technical_claim(
    driver,
    claim_type="host_range",
    claim_text="该工程化构建体成功将宿主范围从KL1型扩展到KL47型肺炎克雷伯菌",
    exact_quote="The engineered phage showed expanded host range to KL47 K. pneumoniae strains",
    claimant_type="publication",
    evidence_context="in_vitro",
    construct_id=construct_id,
    actor_id="test_user"
)
context["claim_ids"]["host_range"] = claim_id_1
print(f"   ✅ 主张1创建成功: {claim_id_1}")

claim_id_2 = create_technical_claim(
    driver,
    claim_type="efficacy",
    claim_text="该构建体对多重耐药肺炎克雷伯菌具有高效裂解活性且安全性良好",
    claimant_type="company",
    evidence_context="in_vitro",
    construct_id=construct_id,
    actor_id="test_user"
)
context["claim_ids"]["efficacy"] = claim_id_2
print(f"   ✅ 主张2创建成功: {claim_id_2}")

print("\n🔬 创建技术结果（Result）...")

result_id_1 = create_technical_result(
    driver,
    result_type="host_range",
    study_context="in_vitro",
    outcome_direction="positive",
    metric_name="host_coverage",
    metric_value=0.85,
    metric_unit="%",
    comparator="亲本噬菌体",
    sample_size=12,
    limitation_summary="仅测试了12株KL47型菌株，需进一步验证",
    reproducibility_status="single_source",
    construct_id=construct_id,
    actor_id="test_user"
)
context["result_ids"]["host_range"] = result_id_1
print(f"   ✅ 结果1创建成功: {result_id_1}")

result_id_2 = create_technical_result(
    driver,
    result_type="lysis",
    study_context="in_vitro",
    outcome_direction="positive",
    metric_name="lysis_efficiency",
    metric_value=98.5,
    metric_unit="%",
    comparator="对照组",
    sample_size=3,
    limitation_summary="仅进行了3次重复实验",
    reproducibility_status="single_source",
    construct_id=construct_id,
    actor_id="test_user"
)
context["result_ids"]["lysis"] = result_id_2
print(f"   ✅ 结果2创建成功: {result_id_2}")

print("\n📊 查询构建体的主张和结果...")
claims = get_claims_by_construct(driver, construct_id)
print(f"📋 主张列表 ({len(claims)} 条)：")
for c in claims:
    print(f"   - {c['claim_type']}: {c['claim_text'][:50]}... (来源: {c['claimant_type']})")

results = get_results_by_construct(driver, construct_id)
print(f"\n📊 结果列表 ({len(results)} 条)：")
for r in results:
    print(f"   - {r['result_type']}: {r['outcome_direction']} | {r['metric_name']}={r['metric_value']}{r['metric_unit']} (n={r['sample_size']})")

print("\n🔍 检测主张-证据缺口...")
gap_analysis = detect_claim_evidence_gaps(driver, construct_id)
if gap_analysis.get('has_gap'):
    print(f"⚠️ 发现 {len(gap_analysis['gaps'])} 个证据缺口：")
    for gap in gap_analysis['gaps'][:3]:
        print(f"   - 主张: {gap['claimed_capability']}")
        print(f"     缺失验证: {gap['missing_validation_stage']}")
        print(f"     严重程度: {gap['gap_severity']}")
else:
    print("   ✅ 所有主张都有对应的实验结果支持")

print("\n🎉 技术主张与证据分离功能测试完成！")

📝 创建技术主张（Claim）...
   ✅ 主张1创建成功: ENG:CLAIM:36ABD8C9
   ✅ 主张2创建成功: ENG:CLAIM:EF753B15

🔬 创建技术结果（Result）...
   ✅ 结果1创建成功: ENG:RESULT:3455008A
   ✅ 结果2创建成功: ENG:RESULT:E09136EB

📊 查询构建体的主张和结果...
📋 主张列表 (2 条)：
   - efficacy: 该构建体对多重耐药肺炎克雷伯菌具有高效裂解活性且安全性良好... (来源: company)
   - host_range: 该工程化构建体成功将宿主范围从KL1型扩展到KL47型肺炎克雷伯菌... (来源: publication)

📊 结果列表 (2 条)：
   - lysis: positive | lysis_efficiency=98.5% (n=3)
   - host_range: positive | host_coverage=0.85% (n=12)

🔍 检测主张-证据缺口...
⚠️ 发现 1 个证据缺口：
   - 主张: 该构建体对多重耐药肺炎克雷伯菌具有高效裂解活性且安全性良好
     缺失验证: in_vitro
     严重程度: high

🎉 技术主张与证据分离功能测试完成！


In [7]:
from config import get_driver
from src.engineering_intelligence.strategy_classifier import get_all_strategies

driver = get_driver()

print("=" * 80)
print("📋 所有实体 ID 汇总")
print("=" * 80)

# 1. 组织
with driver.session() as session:
    print("\n🏢 组织 (Organization):")
    result = session.run("""
        MATCH (o:Organization)
        RETURN o.organization_id AS id, o.canonical_name AS name
        ORDER BY o.canonical_name
    """)
    for r in result:
        print(f"   {r['id']} - {r['name']}")

# 2. 项目
with driver.session() as session:
    print("\n📦 研发项目 (DevelopmentProgram):")
    result = session.run("""
        MATCH (d:DevelopmentProgram)
        RETURN d.program_id AS id, d.canonical_name AS name, d.development_stage AS stage
        ORDER BY d.canonical_name
    """)
    for r in result:
        print(f"   {r['id']} - {r['name']} ({r['stage']})")

# 3. 构建体
with driver.session() as session:
    print("\n🧬 工程化构建体 (EngineeredPhageConstruct):")
    result = session.run("""
        MATCH (ec:EngineeredPhageConstruct)
        OPTIONAL MATCH (ec)-[:IMPLEMENTS]->(es:EngineeringStrategy)
        WITH ec, COLLECT(DISTINCT es.strategy_type) AS strategies
        RETURN ec.construct_id AS id, ec.public_name AS name, ec.construct_status AS status, strategies
        ORDER BY ec.created_at DESC
    """)
    for r in result:
        print(f"   {r['id']} - {r['name'] or '未命名'} ({r['status']}) -> 策略: {', '.join(r['strategies']) if r['strategies'] else '未关联'}")

    print("\n🧬 工程策略:")
    strategies = get_all_strategies(driver)
    for s in strategies:
        print(f"   {s['id']} - {s['strategy_type']}")

📋 所有实体 ID 汇总

🏢 组织 (Organization):
   CI:ORG:31A321BD - Adaptive Phage Therapeutics
   CI:ORG:1D0D468D - Armata Pharmaceuticals
   CI:ORG:D2B4C6DE - BiomX
   CI:ORG:86AA64E6 - Intralytix
   CI:ORG:48615D48 - Locus Biosciences
   CI:ORG:91D05176 - PHAXIAM Therapeutics
   CI:ORG:FCABF250 - Phagelux
   CI:ORG:3AB2C864 - Proteon Pharmaceuticals
   CI:ORG:44078922 - 创噬纪
   CI:ORG:C66F1D9C - 格瑞农生物
   CI:ORG:50563C50 - 青岛诺安百特

📦 研发项目 (DevelopmentProgram):
   CI:PROG:E5FB7E35 - BAFASAL (commercial)
   CI:PROG:4901FEDC - BX004 (phase_2b)
   CI:PROG:B816822F - BX011 (preclinical)
   CI:PROG:2715EABF - BX211 (phase_2)
   CI:PROG:B1B5B460 - LBP-PA01 (phase_1)
   CI:PROG:46F868C1 - ListShield (commercial)
   CI:PROG:D4CFC0BB - PhageBank-CF (phase_1_2)
   CI:PROG:64C63501 - PhageBank-DFO (phase_1_2)
   CI:PROG:5A340ABC - PhageBank-PJI (phase_1_2)
   CI:PROG:A6EC05CC - SalmoFresh (commercial)

🧬 工程化构建体 (EngineeredPhageConstruct):
   ENG:CONST:12FCFA84 - vB_Kpn_HRE_001 (in_vitro_tested) -> 策略: host_ra

In [8]:
# 测试生成 Competitor Brief
from config import get_driver
from src.ci.competitor_brief import generate_competitor_brief
import json

driver = get_driver()

# 从 context 获取 BiomX 的组织 ID（或从列表动态获取）
org_id = context["org_ids"].get("BiomX")
if org_id is None:
    # 如果 context 中没有，则从数据库查询
    from src.ci.competitor_profile import list_organizations
    orgs = list_organizations(driver)
    for org in orgs:
        if org['name'] == 'BiomX':
            org_id = org['id']
            context["org_ids"]["BiomX"] = org_id
            break
    if org_id is None:
        raise ValueError("BiomX 组织未找到，请先导入数据")

brief = generate_competitor_brief(
    driver,
    organization_id=org_id,
    days_back=365
)

print("\n📄 Competitor Brief")
print("="*60)
print(f"组织: {brief['organization']['name']}")
print(f"类型: {brief['organization']['type']}")
print(f"国家: {brief['organization']['country']}")
print(f"数据截止: {brief['as_of_date']}")

print(f"\n📦 项目数量: {len(brief['active_programs'])}")
for prog in brief['active_programs'][:3]:
    print(f"   - {prog['name']} ({prog['stage']}) -> 靶向: {prog['target_pathogens'] or '未指定'}")

print(f"\n📰 近期事件: {len(brief['recent_events'])} 条")
for evt in brief['recent_events'][:3]:
    print(f"   - {evt['date']} | {evt['type']} | {evt['title'][:40]}...")

print(f"\n📊 变化摘要:")
print(f"   新事件: {brief['changes_summary']['new_events']}")
print(f"   高影响事件: {brief['changes_summary']['high_impact_events']}")

if brief['competitive_assessment']['threats']:
    print(f"\n⚠️ 威胁:")
    for t in brief['competitive_assessment']['threats']:
        print(f"   - {t}")

if brief['recommended_next_steps']:
    print(f"\n💡 建议下一步:")
    for step in brief['recommended_next_steps']:
        print(f"   - {step}")

print(f"\n📋 数据缺口:")
for gap in brief['data_gaps']:
    print(f"   - {gap}")

print("\n" + "="*60)
print("完整简报 JSON:")
print(json.dumps(brief, ensure_ascii=False, indent=2))


📄 Competitor Brief
组织: BiomX
类型: biotech
国家: Israel
数据截止: 2026-08-29

📦 项目数量: 3
   - BX004 (phase_2b) -> 靶向: 未指定
   - BX011 (preclinical) -> 靶向: 未指定
   - BX211 (phase_2) -> 靶向: 未指定

📰 近期事件: 3 条
   - 2026-02-03 | partnership | BiomX acquires Adaptive Phage Therapeuti...
   - 2025-11-04 | regulatory_update | BiomX receives positive FDA feedback for...
   - 2025-04-01 | clinical_trial_update | BiomX BX211 Phase 2 positive topline res...

📊 变化摘要:
   新事件: 2
   高影响事件: 1

💡 建议下一步:
   - 建议跟进近期重大事件，评估对内部战略的影响
   - 补充数据缺口，完善竞争对手档案

📋 数据缺口:
   - 公司类型（上市/私有）未确认

完整简报 JSON:
{
  "brief_type": "competitor",
  "as_of_date": "2026-08-29",
  "generated_at": "2026-08-29T21:18:00.041483",
  "organization": {
    "id": "CI:ORG:D2B4C6DE",
    "name": "BiomX",
    "type": "biotech",
    "country": "Israel",
    "website": "https://www.biomx.com",
    "description": "微生物组公司，开发定制化噬菌体疗法靶向慢性疾病中的有害细菌[reference:6]",
    "status": "active",
    "public_or_private": "unknown"
  },
  "active_programs": [
    {
     

In [9]:
# 测试市场情报与工程情报的联动
from config import get_driver
from src.engineering_intelligence.construct_service import (
    link_program_to_construct,
    get_constructs_by_program,
    get_programs_by_strategy
)

driver = get_driver()

# 从 context 读取 ID
construct_id = context["construct_ids"].get("vB_Kpn_HRE_001")
if construct_id is None:
    raise ValueError("构建体未创建")

program_id = context["program_ids"].get("BX004")
if program_id is None:
    # 若 context 没有，则从数据库查询 BX004
    with driver.session() as session:
        result = session.run("""
            MATCH (d:DevelopmentProgram {canonical_name: 'BX004'})
            RETURN d.program_id AS id
        """).single()
        if result:
            program_id = result['id']
            context["program_ids"]["BX004"] = program_id
        else:
            raise ValueError("项目 BX004 未找到")

strategy_id = context["strategy_ids"].get("host_range_expansion")
if strategy_id is None:
    raise ValueError("策略未创建")

print("🔗 建立项目→构建体关联...")
link_program_to_construct(driver, program_id, construct_id)

print(f"\n🔍 查询项目 {program_id} 使用的构建体：")
constructs = get_constructs_by_program(driver, program_id)
for c in constructs:
    print(f"   - {c['name']} ({c['status']})     策略: {c['strategy_type']}     亲本: {c['parent_phage']}")
    print(f"     靶向: {c['target_pathogens']}     主张类型: {c['claim_types']}     结果类型: {c['result_types']}")

print(f"\n🔍 查询使用策略 'host_range_expansion' 的项目：")
progs = get_programs_by_strategy(driver, strategy_id)
for p in progs:
    print(f"   - {p['program_name']} ({p['stage']}) -> 使用构建体: {p['construct_name']}")
    print(f"     靶向: {p['target_pathogens']}")

🔗 建立项目→构建体关联...
✅ 关联创建成功: CI:PROG:4901FEDC → ENG:CONST:12FCFA84

🔍 查询项目 CI:PROG:4901FEDC 使用的构建体：
   - vB_Kpn_HRE_001 (in_vitro_tested)     策略: host_range_expansion     亲本: PKP001
     靶向: ['Klebsiella pneumoniae']     主张类型: ['efficacy', 'host_range']     结果类型: ['lysis', 'host_range']

🔍 查询使用策略 'host_range_expansion' 的项目：
   - BX004 (phase_2b) -> 使用构建体: vB_Kpn_HRE_001
     靶向: []


In [10]:
# 测试 TechnologyAssessment
from config import get_driver
from src.engineering_intelligence.technology_assessment import (
    create_technology_assessment,
    get_assessment_for_subject,
    get_assessments_by_strategy,
    suggest_assessment_from_evidence
)

driver = get_driver()

construct_id = context["construct_ids"].get("vB_Kpn_HRE_001")
if construct_id is None:
    raise ValueError("构建体未创建")
strategy_id = context["strategy_ids"].get("host_range_expansion")
if strategy_id is None:
    raise ValueError("策略未创建")

print("🔍 基于证据自动评估建议：")
suggestion = suggest_assessment_from_evidence(driver, construct_id)
print(f"   构建体: {suggestion['construct_name']}")
print(f"   建议成熟度: {suggestion['suggested_evidence_maturity']}")
print(f"   建议相关性: {suggestion['suggested_technical_relevance']}")
print(f"   有主张: {suggestion['has_claims']}")
print(f"   有结果: {suggestion['has_results']}")
print(f"   {suggestion['note']}")

print("\n📝 创建技术评估...")
assessment_id = create_technology_assessment(
    driver,
    subject_type="construct",
    subject_id=construct_id,
    evidence_maturity=suggestion['suggested_evidence_maturity'],
    technical_relevance=suggestion['suggested_technical_relevance'],
    translational_potential="medium",
    manufacturability_risk="medium",
    safety_uncertainty="low",
    ip_relevance="medium",
    internal_capability_gap="high",
    assessment_summary="基于体外实验数据，该构建体在宿主范围扩展方面表现良好，但在临床转化和可制造性方面存在不确定性。",
    actor_id="test_user"
)
context["tech_assessment_ids"]["construct"] = assessment_id
print(f"   ✅ 评估创建成功: {assessment_id}")

assessment_id_2 = create_technology_assessment(
    driver,
    subject_type="strategy",
    subject_id=strategy_id,
    evidence_maturity="in_vitro",
    technical_relevance="high",
    translational_potential="medium",
    manufacturability_risk="medium",
    safety_uncertainty="medium",
    ip_relevance="medium",
    internal_capability_gap="medium",
    assessment_summary="宿主范围扩展策略在公开文献中证据较多，但该策略的临床转化证据仍然有限。",
    actor_id="test_user"
)
context["tech_assessment_ids"]["strategy"] = assessment_id_2
print(f"   ✅ 策略评估创建成功: {assessment_id_2}")

print("\n🔍 查询构建体的最新评估：")
assessment = get_assessment_for_subject(driver, "construct", construct_id)
if assessment:
    print(f"   评估 ID: {assessment['id']}")
    print(f"   证据成熟度: {assessment['evidence_maturity']}")
    print(f"   技术相关性: {assessment['technical_relevance']}")
    print(f"   转化潜力: {assessment['translational_potential']}")
    print(f"   可制造性风险: {assessment['manufacturability_risk']}")
    print(f"   安全性不确定性: {assessment['safety_uncertainty']}")
    print(f"   内部能力差距: {assessment['internal_capability_gap']}")
    print(f"   摘要: {assessment['summary']}")

print("\n🔍 查询策略 'host_range_expansion' 的相关评估：")
assessments = get_assessments_by_strategy(driver, "host_range_expansion")
for a in assessments:
    print(f"   策略: {a['strategy_type']}")
    print(f"   构建体: {a.get('construct_name', '无')}")
    print(f"   策略成熟度: {a.get('strategy_maturity', '无')}")
    print(f"   构建体成熟度: {a.get('construct_maturity', '无')}")

print("\n🎉 TechnologyAssessment 功能测试完成！")

🔍 基于证据自动评估建议：
   构建体: vB_Kpn_HRE_001
   建议成熟度: in_vitro
   建议相关性: high
   有主张: True
   有结果: True
   此为基于现有证据的自动建议，需专家审核确认

📝 创建技术评估...
   ✅ 评估创建成功: ENG:ASSESS:A8A10AFF
   ✅ 策略评估创建成功: ENG:ASSESS:CBD2C75F

🔍 查询构建体的最新评估：
   评估 ID: ENG:ASSESS:A8A10AFF
   证据成熟度: in_vitro
   技术相关性: high
   转化潜力: medium
   可制造性风险: medium
   安全性不确定性: low
   内部能力差距: high
   摘要: 基于体外实验数据，该构建体在宿主范围扩展方面表现良好，但在临床转化和可制造性方面存在不确定性。

🔍 查询策略 'host_range_expansion' 的相关评估：
   策略: host_range_expansion
   构建体: vB_Kpn_HRE_001
   策略成熟度: in_vitro
   构建体成熟度: in_vitro

🎉 TechnologyAssessment 功能测试完成！


In [15]:
# ================================================================
# 测试: CompetitorAssessment（竞争评估）+ 简报 + 决策（含 Review 节点）
# ================================================================
import warnings
warnings.filterwarnings("ignore")
CONFIG = {
    "organization_id": "",  # 留空自动获取第一个组织
    "assessment_type": "threat",
    "subject_type": "organization",
    "impact_area": "market",
    "impact_level": "high",
    "confidence": "medium",
    "analyst_id": "analyst_zhang",
    "time_horizon": "short",
    "assumptions": ["收购整合顺利", "BX211 临床数据持续向好"],
    "unknowns": ["FDA 对工程噬菌体的审批态度", "收购后产品战略调整"],
    "reviewer_id": "expert_wang",
    "review_decision": "approved",
    "review_comment": "评估逻辑清晰，证据追溯完整，批准作为内部参考。",
    "decision_type": "monitor",
    "decision_owner": "VP_Strategy",
    "review_date": "2027-01-01",
}
# ================================================================

from src.ci.competitor_assessment import (
    create_competitor_assessment,
    get_assessment,
)
from src.shared.review import create_review, get_latest_review
from src.decision_support.decision_record import (
    create_decision_record,
    get_decision_record,
)
from src.ci.competitor_brief import generate_competitor_brief
# 不再需要 update_intelligence_product_review_status

def get_or_pick_organization(driver):
    """从数据库获取第一个组织ID，如果配置中已指定则直接使用"""
    org_id = CONFIG["organization_id"].strip()
    if org_id:
        with driver.session() as session:
            result = session.run(
                "MATCH (o:Organization {organization_id: $oid}) RETURN o",
                oid=org_id
            ).single()
            if result:
                print(f"✅ 使用配置的组织: {org_id}")
                return org_id
            else:
                print(f"⚠️ 配置的组织 {org_id} 不存在，将自动获取第一个组织。")
    # 自动获取第一个组织
    with driver.session() as session:
        result = session.run(
            "MATCH (o:Organization) RETURN o.organization_id AS id, o.canonical_name AS name ORDER BY o.canonical_name LIMIT 1"
        ).single()
        if result:
            print(f"✅ 自动获取组织: {result['id']} ({result['name']})")
            return result['id']
        else:
            raise RuntimeError("数据库中没有组织，请先导入数据。")

# --- 1. 获取组织ID ---
org_id = get_or_pick_organization(driver)
print(f"📌 使用组织ID: {org_id}\n")

# --- 2. 生成一份竞争简报（并审核通过） ---
print("📄 生成竞争简报...")
brief = generate_competitor_brief(driver, org_id, days_back=365, persist=True)
brief_id = brief.get("brief_id")
if not brief_id:
    raise RuntimeError("简报生成失败，无法继续。")
print(f"✅ 简报已生成，ID: {brief_id}")

# ✅ 使用 create_review 创建审核记录（而非直接更新状态）
print("🔍 审核简报（创建 Review 节点）...")
review_brief_id = create_review(
    driver,
    review_type="intelligence_product_review",
    target_object_type="IntelligenceProduct",   # 需要确保 review.py 的 TARGET_ID_MAP 中有 "IntelligenceProduct": "brief_id"
    target_object_id=brief_id,
    reviewer_id=CONFIG["reviewer_id"],
    decision=CONFIG["review_decision"],
    comment="简报内容准确，批准作为决策依据。",
    update_target_status=True,                  # 更新简报的 review_status = approved
    actor_id="system"
)
print(f"✅ 简报审核通过，Review ID: {review_brief_id}")

# --- 3. 创建竞争评估 ---
assess_id = create_competitor_assessment(
    driver,
    assessment_type=CONFIG["assessment_type"],
    subject_type=CONFIG["subject_type"],
    subject_id=org_id,
    impact_area=CONFIG["impact_area"],
    impact_level=CONFIG["impact_level"],
    assessment_summary=f"{org_id} 近期收购 APT，工程化噬菌体管线显著增强，可能在 CRKP 领域形成直接竞争。",
    confidence=CONFIG["confidence"],
    analyst_id=CONFIG["analyst_id"],
    time_horizon=CONFIG["time_horizon"],
    assumptions=CONFIG["assumptions"],
    unknowns=CONFIG["unknowns"],
)
print(f"✅ Step 1: 竞争评估创建成功 (ID: {assess_id})")

assess = get_assessment(driver, assess_id)
print(f"   评估状态: {assess['review_status']}")
print(f"   摘要: {assess['assessment_summary'][:50]}...")

# --- 4. 专家审核评估 ---
review_id = create_review(
    driver,
    review_type="intelligence_product_review",
    target_object_type="CompetitorAssessment",
    target_object_id=assess_id,
    reviewer_id=CONFIG["reviewer_id"],
    decision=CONFIG["review_decision"],
    comment=CONFIG["review_comment"],
    update_target_status=True,
)
print(f"✅ Step 2: 审核通过 (Review ID: {review_id})")

latest = get_latest_review(driver, "CompetitorAssessment", assess_id)
print(f"   审核决策: {latest['decision']} (审核人: {latest['reviewer_id']})")

assess_after = get_assessment(driver, assess_id)
print(f"   评估状态已更新为: {assess_after['review_status']}")

# --- 5. 管理层决策（基于简报） ---
dec_id = create_decision_record(
    driver,
    brief_id=brief_id,               # 必填，传入简报 ID
    decision_type=CONFIG["decision_type"],
    decision_summary=f"将 {org_id} 列入年度重点监控名单，每季度更新管线进展",
    rationale=f"基于竞争评估 {assess_id} 与专家审核 {review_id} 的结论",
    decision_owner=CONFIG["decision_owner"],
    review_date=CONFIG["review_date"],
    # affected_program_id 可选
)
print(f"✅ Step 3: 决策已记录 (Decision ID: {dec_id})")

dec = get_decision_record(driver, dec_id)
print("\n📌 完整决策记录:")
print(f"   决策: {dec['decision_summary']}")
print(f"   负责人: {dec['decision_owner']}")
print(f"   结果状态: {dec['outcome_status']}")
print(f"   关联简报: {dec.get('brief_id') or '无'}")
context["brief_ids"]["test"] = brief_id
context["decision_ids"].append(dec_id)

print("\n🎉 完整决策链: 情报 → 简报 → 评估 → 审核 → 决策")

✅ 自动获取组织: CI:ORG:31A321BD (Adaptive Phage Therapeutics)
📌 使用组织ID: CI:ORG:31A321BD

📄 生成竞争简报...
✅ 简报已生成，ID: CI:BRIEF:6A0F8F05
🔍 审核简报（创建 Review 节点）...
✅ 简报审核通过，Review ID: REV-D733BF1E
✅ Step 1: 竞争评估创建成功 (ID: CI:ASSESS:DCEC9BC7)
   评估状态: draft
   摘要: CI:ORG:31A321BD 近期收购 APT，工程化噬菌体管线显著增强，可能在 CRKP 领域形...
✅ Step 2: 审核通过 (Review ID: REV-28B340AF)
   审核决策: approved (审核人: expert_wang)
   评估状态已更新为: approved
✅ Step 3: 决策已记录 (Decision ID: CI:DEC:11C416E2)

📌 完整决策记录:
   决策: 将 CI:ORG:31A321BD 列入年度重点监控名单，每季度更新管线进展
   负责人: VP_Strategy
   结果状态: pending
   关联简报: CI:BRIEF:6A0F8F05

🎉 完整决策链: 情报 → 简报 → 评估 → 审核 → 决策


In [16]:
# 1. 获取一个有效的组织 ID（修正导入路径）
from src.ci.competitor_brief import generate_competitor_brief

brief_id = context["brief_ids"].get("test")
if brief_id is None:
    # 如果没有，则回退到生成新的（但建议先确保 Cell 11 已执行）
    from src.ci.competitor_profile import list_organizations
    orgs = list_organizations(driver)
    if not orgs:
        raise RuntimeError("数据库中没有组织")
    org_id = orgs[0]['id']
    brief = generate_competitor_brief(driver, org_id, days_back=365)
    brief_id = brief.get("brief_id")
    print(f"⚠️ 未找到 context 中的简报，重新生成: {brief_id}")
else:
    print(f"✅ 使用已审核的简报 ID: {brief_id}")

# 获取最新的决策 ID（如果存在）
dec_id = context["decision_ids"][-1] if context["decision_ids"] else None
if dec_id:
    print(f"✅ 使用关联的决策 ID: {dec_id}")
else:
    print("⚠️ 没有找到关联的决策，将不设置 referenced_decision_id")

# 2. 记录消费事件
from src.ci.use_event_service import record_intelligence_use

use_event_id = record_intelligence_use(
    driver,
    product_id=brief_id,
    consumer_type="IPD",
    consumer_id="ipd_team",
    use_purpose="go_no_go_decision",
    context_note="IPD团队在评估噬菌体工程化产品立项可行性时参考了该简报（已审核通过）。",
    referenced_decision_id=dec_id,   # 关联到 Cell 11 创建的决策
    actor_id="test_user"
)

context["use_event_ids"].append(use_event_id)
print(f"✅ 使用事件记录成功: {use_event_id}")

✅ 使用已审核的简报 ID: CI:BRIEF:6A0F8F05
✅ 使用关联的决策 ID: CI:DEC:11C416E2
✅ 使用事件记录成功: CI:USE:BC70677D


In [19]:
# ==================== Learning Engine 闭环验证，查询完整归因链 ====================
print("=" * 60)
print("Learning Engine 归因链端到端验证")
print("=" * 60)

verification_query = """
MATCH
(src:SourceArtifact)<-[:HAS_SOURCE]-(evt:IntelligenceEvent)-[:AFFECTS]->(prog:DevelopmentProgram)-[:TARGETS_PATHOGEN]->(p:Pathogen),
(brief:IntelligenceProduct)-[:COVERS]->(org:Organization),
(review:Review)-[:REVIEWS]->(brief),
(dec:DecisionRecord)-[:BASED_ON]->(brief),
(use:IntelligenceUseEvent)-[:CONSUMES]->(brief)
WHERE review.decision = 'approved'
RETURN
src.title AS evidence_source,
src.credibility_tier AS credibility,
evt.title AS intelligence_event,
p.species AS target_pathogen,
org.canonical_name AS competitor,
brief.brief_id AS brief_id,
review.decision AS review_status,
dec.decision_type AS decision_type,
use.consumer_type AS consumed_by,
use.use_purpose AS use_purpose
LIMIT 3
"""

with driver.session() as session:
    results = list(session.run(verification_query))

if not results:
    print("❌ 验证失败：归因链不完整，请检查各 P0 需求是否已完成")
else:
    for i, row in enumerate(results, 1):
        print(f"\n— 链路 #{i}")
        print(f"  证据来源 : {row['evidence_source']}")
        print(f"  可信等级 : {row['credibility']}")
        print(f"  情报事件 : {row['intelligence_event']}")
        print(f"  靶向病原 : {row['target_pathogen']}")
        print(f"  竞争对手 : {row['competitor']}")
        print(f"  情报简报 : {row['brief_id']} [审核:{row['review_status']}]")
        print(f"  决策类型 : {row['decision_type']}")
        print(f"  消费方   : {row['consumed_by']} / 用途:{row['use_purpose']}")

    print(f"\n✅ Learning Engine 闭环验证通过 — 共 {len(results)} 条完整链路")
print("=" * 60)

Learning Engine 归因链端到端验证

— 链路 #1
  证据来源 : APT announces FDA IND clearance for PhageBank in DFO
  可信等级 : secondary
  情报事件 : APT announces FDA IND clearance for PhageBank in DFO
  靶向病原 : Klebsiella pneumoniae
  竞争对手 : Adaptive Phage Therapeutics
  情报简报 : CI:BRIEF:6A0F8F05 [审核:approved]
  决策类型 : monitor
  消费方   : IPD / 用途:go_no_go_decision

— 链路 #2
  证据来源 : Proteon Series B funding announcement
  可信等级 : secondary
  情报事件 : Proteon Pharmaceuticals secures €15M Series B funding
  靶向病原 : Salmonella enterica
  竞争对手 : Adaptive Phage Therapeutics
  情报简报 : CI:BRIEF:6A0F8F05 [审核:approved]
  决策类型 : monitor
  消费方   : IPD / 用途:go_no_go_decision

— 链路 #3
  证据来源 : BAFASAL EU Authorization — EMA Decision
  可信等级 : primary
  情报事件 : BAFASAL® receives EU authorization for Salmonella control in poultry
  靶向病原 : Salmonella enterica
  竞争对手 : Adaptive Phage Therapeutics
  情报简报 : CI:BRIEF:6A0F8F05 [审核:approved]
  决策类型 : monitor
  消费方   : IPD / 用途:go_no_go_decision

✅ Learning Engine 闭环验证通过 — 共 3 条完整链路


In [18]:
# 调试：逐项检查链路环节
print("=== 检查各环节是否存在 ===\n")

# 1. SourceArtifact <- HAS_SOURCE - IntelligenceEvent
with driver.session() as session:
    q = "MATCH (src:SourceArtifact)<-[:HAS_SOURCE]-(evt:IntelligenceEvent) RETURN count(*) AS cnt"
    cnt = session.run(q).single()['cnt']
    print(f"1. HAS_SOURCE 关系数量: {cnt}")

# 2. IntelligenceEvent - AFFECTS -> DevelopmentProgram
with driver.session() as session:
    q = "MATCH (evt:IntelligenceEvent)-[:AFFECTS]->(prog:DevelopmentProgram) RETURN count(*) AS cnt"
    cnt = session.run(q).single()['cnt']
    print(f"2. AFFECTS 关系数量: {cnt}")

# 3. DevelopmentProgram - TARGETS_PATHOGEN -> Pathogen
with driver.session() as session:
    q = "MATCH (prog:DevelopmentProgram)-[:TARGETS_PATHOGEN]->(p:Pathogen) RETURN count(*) AS cnt"
    cnt = session.run(q).single()['cnt']
    print(f"3. TARGETS_PATHOGEN 关系数量: {cnt}")

# 4. IntelligenceProduct - COVERS -> Organization
with driver.session() as session:
    q = "MATCH (brief:IntelligenceProduct)-[:COVERS]->(org:Organization) RETURN count(*) AS cnt"
    cnt = session.run(q).single()['cnt']
    print(f"4. COVERS 关系数量: {cnt}")

# 5. Review - REVIEWS -> IntelligenceProduct（且 review_status='approved'）
with driver.session() as session:
    q = """
    MATCH (review:Review)-[:REVIEWS]->(brief:IntelligenceProduct)
    WHERE review.decision = 'approved'
    RETURN count(*) AS cnt
    """
    cnt = session.run(q).single()['cnt']
    print(f"5. REVIEWS (approved) 关系数量: {cnt}")

# 6. DecisionRecord - BASED_ON -> IntelligenceProduct
with driver.session() as session:
    q = "MATCH (dec:DecisionRecord)-[:BASED_ON]->(brief:IntelligenceProduct) RETURN count(*) AS cnt"
    cnt = session.run(q).single()['cnt']
    print(f"6. BASED_ON 关系数量: {cnt}")

# 7. IntelligenceUseEvent - CONSUMES -> IntelligenceProduct
with driver.session() as session:
    q = "MATCH (use:IntelligenceUseEvent)-[:CONSUMES]->(brief:IntelligenceProduct) RETURN count(*) AS cnt"
    cnt = session.run(q).single()['cnt']
    print(f"7. CONSUMES 关系数量: {cnt}")

# 8. 是否存在同时满足所有条件的完整路径
with driver.session() as session:
    q = """
    MATCH (src:SourceArtifact)<-[:HAS_SOURCE]-(evt:IntelligenceEvent)-[:AFFECTS]->(prog:DevelopmentProgram)-[:TARGETS_PATHOGEN]->(p:Pathogen),
          (brief:IntelligenceProduct)-[:COVERS]->(org:Organization),
          (review:Review)-[:REVIEWS]->(brief),
          (dec:DecisionRecord)-[:BASED_ON]->(brief),
          (use:IntelligenceUseEvent)-[:CONSUMES]->(brief)
    WHERE review.decision = 'approved'
    RETURN count(*) AS cnt
    """
    cnt = session.run(q).single()['cnt']
    print(f"8. 完整链路数量: {cnt}")

=== 检查各环节是否存在 ===

1. HAS_SOURCE 关系数量: 11
2. AFFECTS 关系数量: 8
3. TARGETS_PATHOGEN 关系数量: 2
4. COVERS 关系数量: 4
5. REVIEWS (approved) 关系数量: 0
6. BASED_ON 关系数量: 2
7. CONSUMES 关系数量: 2
8. 完整链路数量: 0
